In [85]:
!nvidia-smi

Sun Mar 15 18:25:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             32W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [69]:
!sudo apt update
!sudo apt install -y libopencv-dev pkg-config
!pkg-config --cflags opencv4

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 384 kB in 1s (421 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
126 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Ski

In [70]:
import cv2

In [71]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
!ls '/content/drive/My Drive/Capstone/yolo_custom_model_training'

custom_data	 custom_weight	Makefile    yolov3_custom.cfg
custom_data.zip  darknet	readme.txt


In [73]:
!unzip '/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_data.zip' -d '/content/drive/My Drive/Capstone/yolo_custom_model_training'

Archive:  /content/drive/My Drive/Capstone/yolo_custom_model_training/custom_data.zip
replace /content/drive/My Drive/Capstone/yolo_custom_model_training/custom_data/1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [74]:
!git clone 'https://github.com/AlexeyAB/darknet.git' '/content/drive/My Drive/Capstone/yolo_custom_model_training/darknet'

fatal: destination path '/content/drive/My Drive/Capstone/yolo_custom_model_training/darknet' already exists and is not an empty directory.


In [75]:
%cd /content/drive/My Drive/Capstone/yolo_custom_model_training/darknet

/content/drive/My Drive/Capstone/yolo_custom_model_training/darknet


In [76]:
!ls

3rdparty		darknet_video.py       net_cam_v3.sh
backup			data		       net_cam_v4.sh
build			docker-compose.yml     obj
CCM			Dockerfile.cpu	       package.xml
cfg			Dockerfile.gpu	       README.md
cmake			image_yolov3.sh        results
CMakeLists.txt		image_yolov4.sh        scripts
darknet			include		       src
DarknetConfig.cmake.in	json_mjpeg_streams.sh  vcpkg.json
darknet_images.py	LICENSE		       video_yolov3.sh
darknet.py		Makefile	       video_yolov4.sh


In [77]:
!make

chmod +x *.sh
g++ -std=c++11 -std=c++11 -Iinclude/ -I3rdparty/stb/include -DOPENCV `pkg-config --cflags opencv4 2> /dev/null || pkg-config --cflags opencv` -DGPU -I/usr/local/cuda/include/ -DCUDNN -Wall -Wfatal-errors -Wno-unused-result -Wno-unknown-pragmas -fPIC -rdynamic -Ofast -DOPENCV -DGPU -DCUDNN -I/usr/local/cudnn/include -c ./src/image_opencv.cpp -o obj/image_opencv.o
./src/image_opencv.cpp: In function ‘void draw_detections_cv_v3(void**, detection*, int, float, char**, image**, int, int)’:
./src/image_opencv.cpp:945:23: warning: variable ‘rgb’ set but not used []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-but-set-variable-Wunused-but-set-variable]8;;]
  945 |                 float rgb[3];
      |                       ^~~
./src/image_opencv.cpp: In function ‘void cv_draw_object(image, float*, int, int, int*, float*, int*, int, char**)’:
./src/image_opencv.cpp:1443:14: warning: unused variable ‘buff’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warn

In [78]:
%cd '/content/drive/My Drive/Capstone/yolo_custom_model_training'

/content/drive/My Drive/Capstone/yolo_custom_model_training


In [79]:
# Clean and copy data to writable directory (/tmp)
!rm -rf /tmp/custom_data /tmp/darknet
!cp -r '/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_data' /tmp/custom_data
%cd /tmp

# Run the setup scripts in /tmp
!python custom_data/creating-files-data-and-name.py
!python custom_data/creating-train-and-test-txt-files.py

# Copy darknet to /tmp for training
!cp -r '/content/drive/My Drive/Capstone/yolo_custom_model_training/darknet' /tmp/darknet

# Copy custom cfg file
!cp '/content/drive/My Drive/Capstone/yolo_custom_model_training/yolov3_custom.cfg' /tmp/darknet/cfg/yolov3_custom.cfg

# Verify files were created
!ls -la /tmp/custom_data/classes.names
!ls -la /tmp/custom_data/labelled_data.data
!ls -la /tmp/darknet/cfg/yolov3_custom.cfg

/tmp
-rw-r--r-- 1 root root 8 Mar 15 18:06 /tmp/custom_data/classes.names
-rw-r--r-- 1 root root 120 Mar 15 18:06 /tmp/custom_data/labelled_data.data
-rw------- 1 root root 9117 Mar 15 18:06 /tmp/darknet/cfg/yolov3_custom.cfg


In [80]:
!ls

custom_data
custom_weight
dap_multiplexer.cec9d87ee37f.root.log.INFO.20260315-164207.118
dap_multiplexer.INFO
darknet
debugger_195jj8i17z
directoryprefetcher_binary.cec9d87ee37f.root.log.INFO.20260315-165704.4007
directoryprefetcher_binary.cec9d87ee37f.root.log.INFO.20260315-172105.17872
directoryprefetcher_binary.INFO
drive.cec9d87ee37f.root.log.ERROR.20260315-165657.3812
drive.cec9d87ee37f.root.log.ERROR.20260315-165703.3921
drive.cec9d87ee37f.root.log.ERROR.20260315-172103.17788
drive.cec9d87ee37f.root.log.INFO.20260315-165656.3805
drive.cec9d87ee37f.root.log.INFO.20260315-165656.3812
drive.cec9d87ee37f.root.log.INFO.20260315-165703.3805
drive.cec9d87ee37f.root.log.INFO.20260315-165703.3921
drive.cec9d87ee37f.root.log.INFO.20260315-172103.17781
drive.cec9d87ee37f.root.log.INFO.20260315-172103.17788
drive.cec9d87ee37f.root.log.WARNING.20260315-165657.3812
drive.cec9d87ee37f.root.log.WARNING.20260315-165703.3921
drive.cec9d87ee37f.root.log.WARNING.20260315-172103.17788
drive.ERROR
dri

In [81]:
!chmod +x ./darknet/darknet

In [93]:
%cd /tmp
!mkdir -p /tmp/backup
!mkdir -p /tmp/darknet/backup
!chmod +x /tmp/darknet/darknet
!/tmp/darknet/darknet detector train /tmp/custom_data/labelled_data.data /tmp/darknet/cfg/yolov3_custom.cfg -dont_show -map

/tmp/darknet
backup = /tmp/darknet/backup
 CUDA-version: 12080 (13000), cuDNN: 9.8.0, GPU count: 1  
 OpenCV version: 4.5.4

 Error: There is no custom_data/test.txt file for mAP calculation!
 Don't use -map flag.
 Or set valid=custom_data/train.txt in your /tmp/custom_data/labelled_data.data file. 
Darknet error location: ./src/detector.c, train_detector(), line #37
Error!: No such file or directory
backtrace (8 entries)
1/8: ./darknet(log_backtrace+0x38) [0x5adaa2bdb1f8]
2/8: ./darknet(error+0x3d) [0x5adaa2bdb2dd]
3/8: ./darknet(train_detector+0x37e1) [0x5adaa2c67fc1]
4/8: ./darknet(run_detector+0xa32) [0x5adaa2c69e92]
5/8: ./darknet(main+0x326) [0x5adaa2b9a766]
6/8: /lib/x86_64-linux-gnu/libc.so.6(+0x29d90) [0x7e30c93d4d90]
7/8: /lib/x86_64-linux-gnu/libc.so.6(__libc_start_main+0x80) [0x7e30c93d4e40]
8/8: ./darknet(_start+0x25) [0x5adaa2b9c9e5]


In [94]:
# Verify where Darknet actually saved weights and copy to Drive
!ls -lh /tmp/backup || true
!ls -lh /tmp/darknet/backup || true

# Persist trained weights to Google Drive before runtime ends
!mkdir -p "/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights"
!cp -v /tmp/backup/*.weights "/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/" || true
!cp -v /tmp/darknet/backup/*.weights "/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/" || true
!ls -lh "/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/"

total 1.2G
-rw-r--r-- 1 root root 235M Mar 15 18:17 yolov3_custom_1000.weights
-rw-r--r-- 1 root root 235M Mar 15 18:25 yolov3_custom_2000.weights
-rw-r--r-- 1 root root 235M Mar 15 18:20 yolov3_custom_best.weights
-rw-r--r-- 1 root root 235M Mar 15 18:25 yolov3_custom_final.weights
-rw-r--r-- 1 root root 235M Mar 15 18:25 yolov3_custom_last.weights
total 0
'/tmp/backup/yolov3_custom_1000.weights' -> '/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/yolov3_custom_1000.weights'
'/tmp/backup/yolov3_custom_2000.weights' -> '/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/yolov3_custom_2000.weights'
'/tmp/backup/yolov3_custom_best.weights' -> '/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/yolov3_custom_best.weights'
'/tmp/backup/yolov3_custom_final.weights' -> '/content/drive/My Drive/Capstone/yolo_custom_model_training/custom_weights/yolov3_custom_final.weights'
'/tmp/backup/yolov3_custom_last.weights' -> '/